# Load input data

In [1]:
from functions import get_experiment_data

X_df1, meta_df1, batch_df1, _ = get_experiment_data(
    design_id="df1",
    file_path="/DATA/WGS_study/YSL/projects/Data",
    verbose=True,
)

Design ID               : df1
Design name             : full_raw_norm
Description             : Full raw HIVRC, normalized, standalone LatentGEE
Aggregation             : None
Normalize               : True
Cleanset Filtering      : False
Subset studies          : None
OTU zero-prev           : 0.01
Sample zero-prev        : None
----------------------------------------------------------------------
feature_table           : (1032, 2485)
meta_data               : (1032, 11)
n_batches               : 17


# Trial investigation

In [2]:
import pandas as pd
import glob

# df1 CSV 파일 찾기
# CSV 파일 로드 및 병합
files = [
    "/DATA/WGS_study/YSL/projects/latentgee/examples/logs/df1/phase1/optuna_trials_pid2164806_2026-03-30.csv",
    "/DATA/WGS_study/YSL/projects/latentgee/examples/logs/df1/phase1/optuna_trials_pid4137367_2026-03-31.csv"
]

df_list = [pd.read_csv(f) for f in files]
df1_p1_res = pd.concat(df_list, ignore_index=True)
print(f"전체 trial 수: {len(df1_p1_res)}")

# 필터링
df_clean = df1_p1_res[
    (df1_p1_res['permanova'] > 0.01) &
    (df1_p1_res['permanova'] < 0.1) &
    (df1_p1_res['noise_ratio'] < 0.65) &
    (df1_p1_res['n_clusters'] >= 10)
].sort_values(['noise_ratio', 'permanova', 'n_clusters'], ascending=[True, True, False]).head(20)

print(f"의미있는 trial 수: {len(df_clean)}")
print(df_clean[['trial_number', 'permanova', 'n_clusters', 'noise_ratio', 'cutoff']])

전체 trial 수: 4479
의미있는 trial 수: 20
      trial_number  permanova  n_clusters  noise_ratio  cutoff
412            549   0.052528          13     0.412791    0.01
1614          2064   0.016438          15     0.474806    0.01
3475          4600   0.042887          14     0.475775    0.05
3463          4580   0.016087          17     0.517442    0.03
4268          5574   0.034475          18     0.517442    0.07
658            824   0.010680          18     0.521318    0.01
2296          2923   0.053340          10     0.523256    0.03
1501          1912   0.032648          19     0.526163    0.05
2823          3685   0.029391          13     0.531977    0.01
2737          3562   0.068745          16     0.544574    0.01
316            445   0.019982          19     0.549419    0.05
422            560   0.013176          15     0.551357    0.01
655            821   0.019914          14     0.551357    0.01
340            470   0.029635          13     0.561047    0.01
824           1022   

In [4]:
print(df1_p1_res[df1_p1_res['trial_number'] == 2064])


      cutoff  trial_number  base_dim  n_layers  latent_dim activation  \
1614    0.01          2064       256         2          35       relu   

      strategy  dropout_rate  epochs  learning_rate  ...  min_samples_token  \
1614  constant           0.4     100       0.000841  ...                5.0   

      batch_size             init   beta_kl  weight_decay  grad_clip_norm  \
1614          64  kaiming_uniform  0.010308  5.504142e-08        1.150899   

       csm kl_warmup_ratio       norm  scheduler  
1614  leaf        0.548851  layernorm       none  

[1 rows x 25 columns]


# Best trial selection (trial 2064)

In [ ]:
print(df1_p1_res[df1_p1_res['trial_number'] == 2064].T)

# Run phase 2

In [5]:
from prototype_updated_phase2 import main

df1_p1_res.to_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/logs/df1/phase1/optuna_trials_concat.csv", index=False)
main(
    experiment_name="df1",
    phase=2,
    best_trial_number=2064,
    trial_res_file_phase2="/DATA/WGS_study/YSL/projects/latentgee/examples/logs/df1/phase1/optuna_trials_pid4137367_2026-03-31.csv"
)

2026-05-08 15:59:05 | 로그 저장 경로: /DATA/WGS_study/YSL/projects/latentgee/examples/logs/df1/phase2/latentgee_prototype_cutoff_df1_pid1056555_2026-05-08.log
2026-05-08 15:59:05 | LatentGEE 시작 | experiment=df1 | phase=2
2026-05-08 15:59:05 | config snapshot saved: /DATA/WGS_study/YSL/projects/latentgee/examples/logs/df1/phase2/config_used.yaml
2026-05-08 15:59:05 | python == 3.10.20
2026-05-08 15:59:05 | torch == 2.2.2
2026-05-08 15:59:05 | numpy == 1.26.4
2026-05-08 15:59:05 | scikit-learn == 1.7.2
2026-05-08 15:59:05 | optuna == 3.6.1
2026-05-08 15:59:05 | hdbscan == 0.8.41
Design ID               : df1
Design name             : full_raw_norm
Description             : Full raw HIVRC, normalized, standalone LatentGEE
Aggregation             : None
Normalize               : True
Cleanset Filtering      : False
Subset studies          : None
OTU zero-prev           : 0.01
Sample zero-prev        : None
----------------------------------------------------------------------
feature_table      

In [ ]:
# from functions import zero_filter
def zero_filter(df, meta, cutoff):
    batch="Study"
    prevalence = (df > 0).sum(axis=0) / df.shape[0]
    df = df.loc[:, prevalence > best_cutoff].copy()

    row_sums = df.sum(axis=1)
    keep_sample = row_sums > 0
    df = df.loc[keep_sample].copy()
    meta = meta.loc[keep_sample.values].reset_index(drop=True)
    
    assert len(df) == len(meta)
    assert all(df.index.astype(str) == meta["SeqID"].astype(str))
    
    return df, meta, meta[batch]

best_cutoff = 0.01
X_df1_filtered, meta_df1_filtered, batch_df1_filtered = zero_filter(X_df1, meta_df1, best_cutoff)



In [10]:
X_df1_filtered.to_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/logs/df1/phase2/X_df1_filtered_cutoff0.01.csv", index=True)
meta_df1_filtered.to_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/logs/df1/phase2/meta_df1_filtered_cutoff0.01.csv", index=True)
batch_df1_filtered.to_csv("/DATA/WGS_study/YSL/projects/latentgee/examples/logs/df1/phase2/batch_df1_filtered_cutoff0.01.csv", index=True)

In [8]:
import numpy as np

# X_corrected 로드
X_corrected_df1 = pd.read_csv(
    "/DATA/WGS_study/YSL/projects/latentgee/examples/results/df1/phase2/df1_X_corrected_trial2064_cutoff0.01_2026-05-08.csv",
    index_col=0
)

# inf, NaN 처리
X_corrected_df1_clean = X_corrected_df1.replace([np.inf, -np.inf], np.nan).fillna(0).clip(lower=0)
row_sums = X_corrected_df1_clean.sum(axis=1).replace(0, 1)
X_corrected_df1_clean = X_corrected_df1_clean.div(row_sums, axis=0)

print(f"shape: {X_corrected_df1_clean.shape}")
print(f"NaN 수: {X_corrected_df1_clean.isna().sum().sum()}")
print(f"inf 수: {np.isinf(X_corrected_df1_clean.values).sum()}")

shape: (1032, 2485)
NaN 수: 0
inf 수: 0


In [9]:
from functions import evaluate_batch_correction
df1_result = evaluate_batch_correction(
    X_before     = X_df1_filtered,
    X_after      = X_corrected_df1_clean,
    batch_labels = batch_df1_filtered,
    bio_labels   = meta_df1_filtered['hivstatus'],
    renormalize  = True,
    label        = "df1 — LatentGEE"
)



  df1 — LatentGEE
                        Before   After  Change
Metric                                        
PERMANOVA R² (batch) ↓  0.2092  0.0177 -0.1915
PERMANOVA R² (bio) ↑    0.0026  0.0009 -0.0017
kBET acceptance rate ↑  0.0901  0.9467  0.8566
ASW (batch) → 0        -0.0316 -0.0345 -0.0029
ASW (bio) ↑            -0.0227  0.0004  0.0232

